# Statistical analysis: processed dataset

Analyze engineered **training** features before building a classifier. Validation, refit and test data remain unloaded.
Weekly customer snapshots are correlated: pooled rows are prediction instances, not independent customers.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from online_retail.analysis.processed import (
    feature_summary, class_shares, plot_target_over_time,
    plot_feature_distributions, plot_correlations,
    plot_group_outcomes, plot_weekly_feature_medians,
    target_correlations, plot_target_correlations,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data/processed"
manifest = json.loads((DATA / "manifest.json").read_text())
train = pd.read_parquet(DATA / "train.parquet")
feature_columns = manifest["feature_columns"]
target = manifest["target"]


## Construction checks

Check keys, cutoff cadence, label availability and schema before interpreting plots.
These checks verify the saved dataset; they do not fit preprocessing or models.


In [2]:
assert not train.duplicated(manifest["keys"]).any()
assert train["cutoff"].dt.dayofweek.eq(0).all()
assert train[target].isin([0, 1]).all()
assert (train["cutoff"] + pd.Timedelta(days=manifest["horizon_days"])
        <= pd.Timestamp(manifest["settings"]["validation_start"])).all()
assert set(feature_columns).isdisjoint([*manifest["keys"], target])
assert not train[feature_columns].select_dtypes("number").isin([float("inf"), -float("inf")]).any().any()

pd.Series({"customer_snapshots": len(train), "distinct_customers": train["Customer ID"].nunique(),
           "weekly_cutoffs": train["cutoff"].nunique(),
           "first_cutoff": train["cutoff"].min(), "last_cutoff": train["cutoff"].max()})


/tmp/ipykernel_373121/2194435006.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  assert (train["cutoff"] + pd.Timedelta(days=manifest["horizon_days"])


customer_snapshots                  92319
distinct_customers                   3687
weekly_cutoffs                         33
first_cutoff          2010-03-01 00:00:00
last_cutoff           2010-10-11 00:00:00
dtype: object

## Feature coverage, missingness and ranges

Missing purchase gaps are expected for customers with limited purchase-day history.
Missing is not a zero-day gap. Check unusually large values without automatically
removing them: wholesale customers may legitimately have large baskets.


In [3]:
summary = feature_summary(train, feature_columns)
summary

,dtype,missing_pct,unique_values,min,50%,95%,99%,max
invoice_count,int64,0.000000,107,1.000000,2.000000e+00,9.000000,23.000000,148.000000
purchase_day_count,int64,0.000000,88,1.000000,2.000000e+00,8.000000,18.000000,94.000000
monetary_total,float64,0.000000,10481,1.300000,5.297900e+02,4261.202000,14481.970000,269041.670000
recency_days,float64,0.000000,52194,0.314583,5.161597e+01,206.267778,271.300583,313.586806
observed_history_days,float64,0.000000,65295,0.318056,1.303639e+02,276.500833,304.285042,313.677083
mean_gap_days,float64,46.264583,2073,1.000000,4.500000e+01,140.000000,204.000000,306.000000
std_gap_days,float64,67.226681,4319,0.000000,1.909188e+01,67.175144,106.066017,193.040151
average_invoice_value,float64,0.000000,10445,1.300000,2.926600e+02,940.002857,2013.470000,13280.460000
average_units_per_invoice,float64,0.000000,4928,1.000000,1.500000e+02,600.000000,1359.000000,87167.000000
distinct_product_count,int64,0.000000,573,1.000000,2.900000e+01,154.000000,280.820000,1520.000000


In [4]:
gap_columns = ["last_gap_days", "gap_2_span_days", "gap_3_span_days", "mean_gap_days", "std_gap_days"]
history_bands = pd.cut(train["purchase_day_count"], bins=[0, 1, 2, 3, float("inf")],
                       labels=["1 day", "2 days", "3 days", "4+ days"])
train[gap_columns].isna().groupby(history_bands, observed=True).mean().mul(100).round(1)


,last_gap_days,gap_2_span_days,gap_3_span_days,mean_gap_days,std_gap_days
purchase_day_count,,,,,
1 day,100.0,100.0,100.0,100.0,100.0
2 days,0.0,100.0,100.0,0.0,100.0
3 days,0.0,0.0,100.0,0.0,0.0
4+ days,0.0,0.0,0.0,0.0,0.0


**Interpretation.** Around 46.3% of pooled training snapshots lack a latest purchase-day gap,
and 67.2% lack gap standard deviation. This reflects limited repeat history, rather
than a value to replace with zero. A standard deviation needs at least two completed
gaps (three purchase days). Keep these customers and handle missing values within
training folds later.


## Target balance and change over time

Class 1: purchase in days 0–49; class 0: no purchase within that interval.
Compare pooled balance with weekly balance. Later cutoffs include more customers;
long-inactive customers remain eligible. Accuracy alone may hide a dominant class.


In [5]:
class_counts = train[target].value_counts().reindex([0, 1], fill_value=0).rename("snapshots")
class_table = class_counts.to_frame()
class_table["share_pct"] = class_counts / len(train) * 100
class_table


,snapshots,share_pct
RepurchaseWithin50Days,,
0,55824,60.468593
1,36495,39.531407


In [6]:
plot_target_over_time(train, target)
plt.show()


/tmp/ipykernel_373121/2296781219.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** About 39.5% of training snapshots have a purchase within 50 days;
60.5% do not. A constant non-purchase prediction reaches 60.5% training accuracy.
Weekly shares reveal changes in outcome prevalence; repeated customer weeks are correlated.

## Distributions for one customer population

Use the last training cutoff (one row per customer). Invoice count and spending
use base-10 logarithmic x-axes with ticks in original units (1, 10, 100, 1,000...).
Their bins are equally spaced on the log axis: each covers the same multiplicative
range, not the same absolute width. Bar heights show customer counts, not density.
Recency, observed history, purchase gaps and recent invoice counts stay linear.
In particular, recent counts retain meaningful zeros; zero cannot lie on a log axis.
Missing gaps are omitted, not replaced by zero. No model features are transformed.


In [7]:
important_features = [
    "recency_days", "invoice_count", "monetary_total",
    "observed_history_days", "invoice_count_30d", "last_gap_days",
]
last_cutoff = train["cutoff"].max()
latest_train = train.loc[train["cutoff"].eq(last_cutoff)]
plot_feature_distributions(latest_train, important_features)
plt.show()


/tmp/ipykernel_373121/3863128265.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation (2010-10-11).** The median customer has 2 invoices and recorded spend
of £631.79; the 99th percentiles are 28 invoices and about £18,068. The long right
tails explain the logarithmic axes: a few large buyers should not compress the bulk
of customers into the first bar. Large values alone do not establish errors.

About 65.8% of these customers have no invoice in the preceding 30 days. Median
recency is about 62 days. The median latest completed gap is 57 days **among customers
with a defined gap**; it does not describe the return time of all customers.


## Predictor relationships

Show the six core features plus recent spend and the combined RFM score.
Annotated Spearman correlations describe monotonic associations using pairwise
nonmissing rows. RFM score is derived from other predictors, so some correlation
is expected. Correlation is not model importance or a reason to automatically
remove a feature. Other predictors remain in the saved datasets.


In [8]:
relationship_features = important_features + ["monetary_30d", "rfm_score"]
plot_correlations(train, relationship_features)
plt.show()


/tmp/ipykernel_373121/701359008.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** Historical invoice count and spend are strongly associated
(Spearman ≈ 0.76). Recent invoice count and recent spend are even more closely
associated (≈ 0.98), partly because both are zero for inactive customers. Recency
is strongly negatively associated with recent invoice count (≈ −0.82), as expected.
RFM score is derived from its components, so its correlations are not new independent
evidence. Retain these candidates until chronological model comparisons establish
whether removing redundant features helps.


## Associations with each target class

Each target column is an indicator: 1 when that class occurred, 0 otherwise.
Positive correlation means larger feature values tend to accompany that outcome;
negative correlation means the opposite. The two binary indicators are complements;
their correlations have opposite signs. Missing features are excluded pairwise; see available-row counts.
These are pooled training associations across correlated weekly snapshots, not
causal effects, independent significance tests, or out-of-time model performance.


In [9]:
outcome_correlations = target_correlations(train, relationship_features, target)
plot_target_correlations(outcome_correlations)
plt.show()


/tmp/ipykernel_373121/3629194230.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# Counts clarify that gap correlations describe a subset with observed repeat history.
train[relationship_features].notna().sum().rename("available_snapshots").to_frame()


,available_snapshots
recency_days,92319
invoice_count,92319
monetary_total,92319
observed_history_days,92319
invoice_count_30d,92319
last_gap_days,49608
monetary_30d,92319
rfm_score,92319


**Interpretation.** Compare feature associations with purchase versus non-purchase.
The two outcome columns are complements, not separate timing outcomes. Missing-gap
associations describe repeat-history snapshots only. Associations are not campaign effects.

## Historical associations with outcomes

Bars below describe pooled training snapshots, not classifier predictions.
Compare sparse histories, recency groups, and relative RFM segments. These patterns
are not campaign effects. Feature summaries by class are descriptive; avoid p-values
that treat repeated customer-weeks as independent observations.


In [11]:
plot_group_outcomes(train, target)
plt.show()


/tmp/ipykernel_373121/407907390.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Show denominators alongside conditional outcome percentages.
rfm_outcomes = class_shares(train, target, "rfm_segment")
rfm_outcomes.insert(0, "snapshots", train.groupby("rfm_segment").size())
rfm_outcomes.reindex(["low", "medium", "high"]).round(2)


RepurchaseWithin50Days,snapshots,0,1
rfm_segment,,,
low,54461,73.77,26.23
medium,18216,54.89,45.11
high,19642,28.75,71.25


**Interpretation.** Frequent/recent profiles and higher RFM segments show larger
near-term purchase shares than sparse/inactive profiles. This supports evaluating
performance separately for one-invoice customers. Group differences do not prove
that an RFM score improves a classifier beyond continuous RFM inputs, or that an
offer would cause a purchase. Segment counts are customer-week counts.


In [13]:
comparison_columns = ["recency_days", "invoice_count", "monetary_total",
                      "observed_history_days", "invoice_count_30d", "monetary_30d",
                      "last_gap_days", "rfm_score"]
train.groupby(target)[comparison_columns].median().T


RepurchaseWithin50Days,0,1
recency_days,66.619444,34.450694
invoice_count,1.000000,3.000000
monetary_total,369.250000,934.200000
observed_history_days,122.326389,143.386111
invoice_count_30d,0.000000,0.000000
monetary_30d,0.000000,0.000000
last_gap_days,56.000000,38.000000
rfm_score,33.333333,66.666667


## Evolution of cumulative features

All-history totals and observed exposure tend to grow as the source accumulates
history. This is not automatically a seasonal effect. Calendar features have one
value per week: thousands of customer rows do not create thousands of independent
observations of a season. Training covers only part of an annual cycle.


In [14]:
plot_weekly_feature_medians(train)
plt.show()

/tmp/ipykernel_373121/3530432277.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** From the first to the last training cutoff, median observed
history increases from about 74 to 216 days, invoices from 1 to 2, and spend from
£399 to £632. Median recency also increases from about 32 to 62 days. Accumulating
history and retaining inactive customers can change these distributions; these
trends are not sufficient evidence of annual seasonality.


## Initial training observations

- Purchase share is approximately 39.53%; non-purchase share is 60.47%. A constant non-purchase prediction reaches 60.47% training accuracy; this is not a validated model result.
- One-invoice and long-inactive snapshots show higher non-purchase shares than frequent/recent profiles. This supports evaluating sparse-history customers separately.
- Higher relative RFM segments show more near-term purchases in training. This does not establish that the combined score improves predictions beyond continuous RFM features.
- Review the missingness table and weekly curves before choosing how to handle sparse histories or cumulative-feature growth. No predictors have been removed.


## Multicollinearity and feature redundancy

Multicollinearity concerns a numeric model design matrix. We inspect numeric
correlations and variance inflation factors (VIF), plus categorical and mixed
associations. These are training-only diagnostics, not automatic removal rules.
No target is included. Repeated customer-weeks preclude naive significance tests.


In [15]:
from online_retail.analysis.collinearity import (
    correlated_pairs, encoded_vif, categorical_associations,
    mixed_associations, plot_associations,
)

# Full predictor inventory, not just the small visualization subset above.
strong_pairs = correlated_pairs(train, feature_columns, threshold=0.8)
strong_pairs.round(3)


,feature_a,feature_b,pearson,spearman
0,invoice_count,purchase_day_count,0.949,0.974
1,invoice_count,invoice_count_90d,0.851,0.628
2,purchase_day_count,invoice_count_90d,0.801,0.613
3,monetary_total,monetary_90d,0.903,0.667
4,monetary_total,m_score,0.212,0.861
5,recency_days,invoice_count_30d,-0.446,-0.815
6,recency_days,monetary_30d,-0.202,-0.803
7,recency_days,invoice_count_90d,-0.427,-0.810
8,recency_days,r_score,-0.840,-0.771
9,mean_gap_days,last_gap_days,0.899,0.851


### VIF on the encoded design

VIF regresses each predictor on all the others: 1 means no linear redundancy;
large values indicate unstable coefficient allocation. Values above 5 or 10
are review flags, not universal cutoffs. Infinity denotes an effectively exact
linear dependency under a numerical tolerance.

For this diagnostic only, numeric missing values are median-filled and categorical
variables (including booleans) are reference-coded with one level dropped. The
intercept is handled by each auxiliary regression. Standardization improves numerical
conditioning. Constant/all-missing columns are reported separately. Model preprocessing
will be fitted independently within each chronological training fold.


In [16]:
vif_all, excluded_vif_columns = encoded_vif(train, feature_columns)
print("Constant/all-missing encoded columns excluded:", excluded_vif_columns)
vif_all.round(2)


Constant/all-missing encoded columns excluded: []


,feature,vif,r_squared_other_features
20,r_score,inf,1.00
21,f_score,inf,1.00
22,m_score,inf,1.00
23,rfm_score,inf,1.00
26,rfm_segment_low,31.45,0.97
0,invoice_count,20.82,0.95
19,days_until_christmas,17.88,0.94
1,purchase_day_count,15.43,0.94
17,month_sin,15.09,0.93
4,observed_history_days,14.07,0.93


**Interpretation.** R/F/M component scores and their average have infinite VIF,
as expected from their exact linear relationship. Keeping all four makes individual
linear coefficients non-identifiable without regularization. Reference coding
prevents the ordinary dummy-variable trap, but cannot remove dependencies between
different feature representations.


### Categorical and mixed associations

The manifest contains one string category, `rfm_segment`, plus two boolean history
indicators. We also inspect the discrete R/F/M scores as ordered categories here.
Bias-corrected Cramer's V describes categorical association (0–1); it is not VIF.
Correlation ratio eta describes how much numeric variation is explained by a
categorical grouping (0–1); it does not establish linear dependence.

Both diagnostics summarize pooled training snapshots without p-values. A high
association can arise by construction—for example, RFM segment is derived from
scores, and history flags are derived from purchase-day counts.


In [17]:
categorical_columns = ["rfm_segment", "has_previous_purchase_day", "has_four_purchase_days",
                       "r_score", "f_score", "m_score"]
category_association = categorical_associations(train, categorical_columns)
plot_associations(category_association, "Categorical associations — corrected Cramer's V")
plt.show()


/tmp/ipykernel_373121/1167722287.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** RFM segment is strongly associated with F score (Cramer's V
≈ 0.94), and F score is strongly associated with the four-purchase-day flag
(≈ 0.94). These features summarize related aspects of repeat activity. Strong
association alone is not exact linear dependence, and is not a reason to remove
both representations from every model.


In [18]:
mixed_numeric_columns = ["recency_days", "invoice_count", "monetary_total",
                         "purchase_day_count", "last_gap_days", "rfm_score"]
mixed_association = mixed_associations(train, categorical_columns, mixed_numeric_columns)
plot_associations(mixed_association, "Categorical–numeric associations — correlation ratio eta")
plt.show()


/tmp/ipykernel_373121/161951241.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Compare without the derived RFM representation

`rfm_score = (r_score + f_score + m_score) / 3` is an exact linear dependency.
Its low/medium/high segment is another representation of the same score.
For the first logistic baseline, use continuous behavioral features and omit all
five derived RFM columns. This is a representation choice, not a declaration that
RFM segmentation is useless. Compare an RFM-augmented variant later within training.

The following second VIF table shows remaining redundancy without changing saved
datasets. Elastic net may shrink redundant coefficients, but does not guarantee
stable feature selection. Trees usually tolerate correlated predictors better,
although importance attribution can still be shared or unstable.


In [19]:
derived_rfm = ["r_score", "f_score", "m_score", "rfm_score", "rfm_segment"]
continuous_baseline_columns = [c for c in feature_columns if c not in derived_rfm]
vif_without_rfm, _ = encoded_vif(train, continuous_baseline_columns)
vif_without_rfm.round(2)


,feature,vif,r_squared_other_features
0,invoice_count,20.46,0.95
19,days_until_christmas,17.87,0.94
17,month_sin,15.06,0.93
1,purchase_day_count,15.05,0.93
4,observed_history_days,13.32,0.92
13,monetary_90d,11.38,0.91
12,invoice_count_90d,10.39,0.90
2,monetary_total,9.76,0.90
3,recency_days,8.79,0.89
5,mean_gap_days,8.33,0.88


**Interpretation.** After omitting derived RFM, invoice count still has VIF ≈ 20.5
and purchase-day count ≈ 15.0: the two capture overlapping purchase frequency.
Calendar features also overlap over this limited training period (days until
Christmas ≈ 17.9; month sine ≈ 15.1). Observed history has VIF ≈ 13.3, reflecting
its relationship with accumulated totals and calendar progression. These diagnostics
support shrinkage and feature-group comparisons; they do not establish which
predictors improve future log loss. Saved datasets remain unchanged.
